---
# 🔧 STEP 1: Setup and Installation

In [ ]:
Yeh project ek interactive AI News App ki tarah behave karega

Real Example Flow – User Experience
Step 1: User App Open Karega

Gradio app open hoti hai browser mai — title likha hoga:

 News Classifier + Summarizer

Wahan ek text box dikhai deta hai:

“Paste your news article here…

In [ ]:
⚙️ Step 3: System Kaam Shuru Karta Hai

DistilBERT Classifier input text padhta hai
→ predict karta hai ye kis topic ki news hai

BART Summarizer same text se chhoti 2–3 line ki summary banata hai

Yeh dono kaam Hugging Face Transformers se hote hain.

In [ ]:
system_prompt = Pakistan won the Asia Cup 2025 after defeating India by five wickets in a thrilling final match held in Dubai. The captain credited the bowlers for setting up the win and praised the fans for their support.


In [ ]:
# ================================================================
# STEP 1A: Check if GPU is available
# ================================================================
import torch

if torch.cuda.is_available():
    print(f"✅ GPU Available: {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  GPU not found — CPU use hoga (thoda slow hoga, but chalega!)")
    print("   Tip: Runtime → Change Runtime Type → T4 GPU")

✅ GPU Available: Tesla T4
   GPU Memory: 15.6 GB


In [ ]:
# ================================================================
# STEP 1B: Install all required libraries
# ================================================================
# Sirf pehli baar run karna hai — 1-2 minutes lagenge

!pip install transformers datasets gradio sentencepiece accelerate -q

print("✅ All libraries installed!")

✅ All libraries installed!


In [ ]:
# ================================================================
# STEP 1C: Import all libraries
# ================================================================
import torch
import gradio as gr
import warnings
import time
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForSequenceClassification
)
import torch.nn.functional as F

warnings.filterwarnings('ignore')

# Device set karo
# 0 = GPU, -1 = CPU (HuggingFace pipeline convention)
DEVICE = 0 if torch.cuda.is_available() else -1

print("✅ Libraries imported successfully!")
print(f"   Running on: {'GPU 🚀' if DEVICE == 0 else 'CPU 🐢'}")

✅ Libraries imported successfully!
   Running on: GPU 🚀


---
# 📚 STEP 2: Load Models

### 🔑 Key Concept — Kya Farq Hai Dono Models Mein?

```
DistilBERT (Encoder Only)
  Text → [BERT Encoder] → Single Label Output
  Samajhta hai, generate nahi karta

BART (Encoder + Decoder)
  Text → [Encoder] → [Decoder] → New Text Output
  Samajhta bhi hai, naya text generate bhi karta hai
```

> ⏳ **Models download hone mein 2-3 minute lag sakte hain (first time only)**

In [ ]:
Label	Category	Example
LABEL_0	🌍 World	“UN meeting held in New York on climate change.”
LABEL_1	🏏 Sports	“Pakistan defeats India in a close final match.”
LABEL_2	💰 Business	“Stock markets hit record highs this week.”
LABEL_3	💻 Sci/Tech	“Apple launches new AI-powered iPhone.”

In [ ]:
# ================================================================
# STEP 2A: Load News Classifier (DistilBERT fine-tuned on AG News)
# ================================================================
# AG News Dataset ke 4 categories:
# LABEL_0 = World, LABEL_1 = Sports, LABEL_2 = Business, LABEL_3 = Sci/Tech

print("📥 Loading News Classifier (DistilBERT)...")
print("   Model: fabriceyhc/bert-base-uncased-ag_news")

start = time.time()

classifier = pipeline(
    task="text-classification",
    model="fabriceyhc/bert-base-uncased-ag_news",
    device=DEVICE
)

print(f"✅ Classifier loaded! ({time.time()-start:.1f}s)")

📥 Loading News Classifier (DistilBERT)...
   Model: fabriceyhc/bert-base-uncased-ag_news


config.json:   0%|          | 0.00/919 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: fabriceyhc/bert-base-uncased-ag_news
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

tokenizer_config.json:   0%|          | 0.00/321 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

✅ Classifier loaded! (10.8s)


In [ ]:
# ================================================================
# STEP 2B: Load Summarizer (BART — Primary Model)
# ================================================================
# facebook/bart-large-cnn — CNN/DailyMail news pe fine-tuned
# Agar RAM error aaye toh neeche wala backup cell run karo

print("📥 Loading Summarizer (BART)...")
print("   Model: facebook/bart-large-cnn")
print("   ⚠️  Yeh model ~1.6GB ka hai — thoda time lagega")

start = time.time()

from transformers import BartForConditionalGeneration, BartTokenizer

tokenizer = BartTokenizer.from_pretrained("facebook/bart-large-cnn")
model = BartForConditionalGeneration.from_pretrained("facebook/bart-large-cnn")
model.to(DEVICE if DEVICE != -1 else "cpu") # Move model to GPU if available

def custom_summarizer(text_inputs, max_length=100, min_length=30, do_sample=False):
    inputs = tokenizer(text_inputs, return_tensors="pt", truncation=True, max_length=1024)
    if DEVICE != -1: # Move inputs to GPU if model is on GPU
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    summary_ids = model.generate(
        inputs['input_ids'],
        num_beams=4,
        max_length=max_length,
        min_length=min_length,
        early_stopping=True,
        do_sample=do_sample
    )
    # The pipeline returns a list of dictionaries with 'summary_text' key
    return [{'summary_text': tokenizer.decode(s_id, skip_special_tokens=True)} for s_id in summary_ids]

summarizer = custom_summarizer # Assign the custom function to 'summarizer'

print(f"✅ Summarizer loaded! ({time.time()-start:.1f}s)")
print("\n🎉 Both models ready! Ab next cells run karo.")

📥 Loading Summarizer (BART)...
   Model: facebook/bart-large-cnn
   ⚠️  Yeh model ~1.6GB ka hai — thoda time lagega


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

✅ Summarizer loaded! (17.3s)

🎉 Both models ready! Ab next cells run karo.


In [ ]:
# ================================================================
# STEP 2B — BACKUP: Agar BART RAM error de toh T5-Small use karo
# ================================================================
# Sirf tab run karo jab BART wala cell fail ho jaye
# Uncomment karke run karo

# print("📥 Loading Backup Summarizer (T5-Small)...")
# summarizer = pipeline(
#     task="summarization",
#     model="t5-small",
#     device=DEVICE
# )
# print("✅ T5-Small summarizer loaded!")

---
# 🧪 STEP 3: Test Each Model Individually

### Pehle alag alag test karte hain — phir combine karenge

In [ ]:
# ================================================================
# STEP 3A: Test the Classifier alone
# ================================================================

# AG News label mapping — model LABEL_0, LABEL_1 etc. return karta hai
LABEL_MAP = {
    "LABEL_0": "🌍 World News",
    "LABEL_1": "🏆 Sports",
    "LABEL_2": "💼 Business",
    "LABEL_3": "💻 Sci/Tech"
}

# 4 test articles — ek ek category ke liye
test_articles = [
    "Apple unveiled its latest iPhone with advanced AI chips and improved camera. The new device features neural processing capabilities for on-device machine learning.",
    "Pakistan won the cricket match against India by 5 wickets in the Asia Cup final yesterday in Colombo.",
    "The Federal Reserve raised interest rates by 25 basis points to combat rising inflation in the US economy this quarter.",
    "The United Nations held an emergency session to discuss the ongoing conflict in the Middle East region and possible ceasefire options."
]

print("🔍 CLASSIFIER TEST RESULTS")
print("=" * 60)

for i, article in enumerate(test_articles):
    result = classifier(article[:512])[0]  # 512 token limit hai BERT ki
    label = LABEL_MAP.get(result['label'], result['label'])
    confidence = result['score'] * 100

    print(f"\n📰 Article {i+1}: {article[:65]}...")
    print(f"   ✅ Predicted : {label}")
    print(f"   📊 Confidence: {confidence:.1f}%")

🔍 CLASSIFIER TEST RESULTS

📰 Article 1: Apple unveiled its latest iPhone with advanced AI chips and impro...
   ✅ Predicted : 💻 Sci/Tech
   📊 Confidence: 97.6%

📰 Article 2: Pakistan won the cricket match against India by 5 wickets in the ...
   ✅ Predicted : 🌍 World News
   📊 Confidence: 89.2%

📰 Article 3: The Federal Reserve raised interest rates by 25 basis points to c...
   ✅ Predicted : 💼 Business
   📊 Confidence: 99.7%

📰 Article 4: The United Nations held an emergency session to discuss the ongoi...
   ✅ Predicted : 🌍 World News
   📊 Confidence: 99.9%


In [ ]:
# ================================================================
# STEP 3B: Test the Summarizer alone
# ================================================================

long_article = """
Artificial intelligence has made significant strides in recent years,
transforming industries from healthcare to finance. Machine learning models
now power everything from recommendation systems to autonomous vehicles.
Tech giants like Google, Microsoft, and Amazon are investing billions into
AI research. OpenAI's ChatGPT reached 100 million users in just two months,
making it the fastest-growing consumer application in history. Meanwhile,
researchers are raising concerns about AI safety and the potential risks of
advanced AI systems. Governments around the world are now working on
regulatory frameworks to ensure responsible AI development. The European Union
passed the AI Act — the world's first comprehensive AI regulation — which
categorizes AI applications by risk level and imposes strict requirements
on high-risk systems. Experts believe that while AI presents enormous
opportunities, society must carefully manage the transition to ensure
benefits are distributed fairly and risks are minimized.
"""

print("📰 ORIGINAL ARTICLE:")
print("-" * 60)
print(long_article.strip())
print(f"\n   Word Count: {len(long_article.split())} words")

print("\n⏳ Generating summary...")
summary_result = summarizer(
    long_article,
    max_length=80,
    min_length=25,
    do_sample=False   # Greedy decoding — deterministic output
)
summary_text = summary_result[0]['summary_text']

print("\n✅ SUMMARY:")
print("-" * 60)
print(summary_text)
print(f"\n   Summary Word Count  : {len(summary_text.split())} words")
print(f"   Compression Ratio   : {len(long_article.split()) / len(summary_text.split()):.1f}x shorter")

📰 ORIGINAL ARTICLE:
------------------------------------------------------------
Artificial intelligence has made significant strides in recent years,
transforming industries from healthcare to finance. Machine learning models
now power everything from recommendation systems to autonomous vehicles.
Tech giants like Google, Microsoft, and Amazon are investing billions into
AI research. OpenAI's ChatGPT reached 100 million users in just two months,
making it the fastest-growing consumer application in history. Meanwhile,
researchers are raising concerns about AI safety and the potential risks of
advanced AI systems. Governments around the world are now working on
regulatory frameworks to ensure responsible AI development. The European Union
passed the AI Act — the world's first comprehensive AI regulation — which
categorizes AI applications by risk level and imposes strict requirements
on high-risk systems. Experts believe that while AI presents enormous
opportunities, society must caref

---
# 🔗 STEP 4: Combine Both Models — Full Pipeline

In [ ]:
# ================================================================
# STEP 4A: Build the combined pipeline function
# ================================================================

def analyze_news(article_text):
    """
    Full News Analysis Pipeline
    ----------------------------
    Input  : Raw news article text (string)
    Output : (category, confidence, summary)

    Step 1 → DistilBERT classifies the article
    Step 2 → BART generates a concise summary
    """

    # --- Input validation ---
    if not article_text or len(article_text.strip()) < 50:
        return "❌ Error", "0%", "Please provide at least 50 characters of news text."

    # --- Step 1: Classify ---
    # BERT ki 512 token limit hai — isliye pehle 512 characters lenge
    classification_result = classifier(article_text[:512])[0]
    raw_label  = classification_result['label']
    category   = LABEL_MAP.get(raw_label, raw_label)
    confidence = f"{classification_result['score'] * 100:.1f}%"

    # --- Step 2: Summarize ---
    # BART ki input limit ~1024 tokens — 1000 chars safe limit
    text_for_summary = article_text[:1000]
    summary_result   = summarizer(
        text_for_summary,
        max_length=100,
        min_length=30,
        do_sample=False
    )
    summary = summary_result[0]['summary_text']

    return category, confidence, summary


print("✅ Pipeline function ready!")
print("\n📖 Usage:")
print("   category, confidence, summary = analyze_news(your_article)")

✅ Pipeline function ready!

📖 Usage:
   category, confidence, summary = analyze_news(your_article)


In [ ]:
# ================================================================
# STEP 4B: Test the full combined pipeline
# ================================================================

test_news = """
Tesla reported record quarterly profits as electric vehicle sales surged
across North America and Europe. CEO Elon Musk announced plans to build
three new Gigafactories in Asia to meet growing demand. The company's
stock rose 8% following the earnings report. Tesla also revealed a new
self-driving software update that improves highway autopilot performance
by 40%. Analysts predict Tesla will maintain market leadership in the EV
sector through 2025, despite growing competition from Ford, GM, and
Chinese manufacturers like BYD and NIO.
"""

print("🧪 FULL PIPELINE TEST")
print("=" * 60)
print("INPUT ARTICLE:")
print(test_news.strip())

print("\n⏳ Processing through pipeline...")
category, confidence, summary = analyze_news(test_news)

print("\n" + "=" * 60)
print("✅ PIPELINE RESULTS")
print("=" * 60)
print(f"📌 Category   : {category}")
print(f"📊 Confidence : {confidence}")
print(f"\n📝 Summary:")
print(f"   {summary}")
print("=" * 60)

🧪 FULL PIPELINE TEST
INPUT ARTICLE:
Tesla reported record quarterly profits as electric vehicle sales surged
across North America and Europe. CEO Elon Musk announced plans to build
three new Gigafactories in Asia to meet growing demand. The company's
stock rose 8% following the earnings report. Tesla also revealed a new
self-driving software update that improves highway autopilot performance
by 40%. Analysts predict Tesla will maintain market leadership in the EV
sector through 2025, despite growing competition from Ford, GM, and
Chinese manufacturers like BYD and NIO.

⏳ Processing through pipeline...

✅ PIPELINE RESULTS
📌 Category   : 💻 Sci/Tech
📊 Confidence : 95.5%

📝 Summary:
   Tesla reported record quarterly profits as electric vehicle sales surged. CEO Elon Musk announced plans to build three new Gigafactories in Asia to meet growing demand. Tesla also revealed a new self-driving software update.


---
# 📊 STEP 5: Batch Test — Multiple Articles at Once

In [ ]:
# ================================================================
# STEP 5: Run all 4 categories through the pipeline
# ================================================================

sample_articles = {
    "Tech Article": """
    Google has launched its new Gemini Ultra AI model, claiming it surpasses
    GPT-4 in multiple benchmarks. The model can process text, images, audio,
    and video simultaneously. Google integrated Gemini into its search engine
    and Workspace apps. Early tests show impressive performance on coding,
    reasoning, and multilingual tasks. Developers can access Gemini via API
    through Google Cloud.
    """,

    "Sports Article": """
    Pakistan cricket team secured a thrilling victory in the ICC World Cup
    semi-final against Australia by 6 wickets. Babar Azam scored a brilliant
    century while Shaheen Afridi claimed four wickets to restrict Australia
    to 245 runs. The win sets up a final clash against India. Captain Babar
    praised the team performance saying they executed the plan perfectly
    under pressure in difficult conditions.
    """,

    "Business Article": """
    The State Bank of Pakistan raised interest rates by 100 basis points to
    control inflation which reached 28% last month. The decision impacted
    the Pakistani rupee which gained 2% against the US dollar. The stock
    market reacted positively with the KSE-100 index rising 500 points.
    Economists predict the inflation will ease to 15% by next quarter if
    oil prices remain stable in global markets.
    """,

    "World News Article": """
    The United Nations Security Council held an emergency session to address
    the escalating conflict in Eastern Europe. World leaders called for an
    immediate ceasefire and diplomatic negotiations. The US and EU announced
    new sanctions while humanitarian aid corridors were established for
    civilians. The UN Secretary-General urged both sides to prioritize
    civilian protection and allow access to conflict zones.
    """
}

print("📊 BATCH ANALYSIS RESULTS")
print("=" * 70)

for title, article in sample_articles.items():
    category, confidence, summary = analyze_news(article)
    print(f"\n{'─' * 70}")
    print(f"📰 {title}")
    print(f"{'─' * 70}")
    print(f"📌 Category   : {category}")
    print(f"📊 Confidence : {confidence}")
    print(f"📝 Summary    : {summary}")

print(f"\n{'=' * 70}")
print("✅ Batch analysis complete!")

📊 BATCH ANALYSIS RESULTS

──────────────────────────────────────────────────────────────────────
📰 Tech Article
──────────────────────────────────────────────────────────────────────
📌 Category   : 💻 Sci/Tech
📊 Confidence : 96.5%
📝 Summary    : Google has launched its new Gemini Ultra AI model. The model can process text, images, audio, and video simultaneously. Early tests show impressive performance on coding, reasoning, and multilingual tasks.

──────────────────────────────────────────────────────────────────────
📰 Sports Article
──────────────────────────────────────────────────────────────────────
📌 Category   : 🏆 Sports
📊 Confidence : 100.0%
📝 Summary    : Pakistan beat Australia by 6 wickets in the ICC World Cup semi-final. Babar Azam scored a brilliant century while Shaheen Afridi claimed four wickets. The win sets up a final clash against India.

──────────────────────────────────────────────────────────────────────
📰 Business Article
─────────────────────────────────────────

---
# 🌐 STEP 6: Gradio Web App — Live UI!

> 🎉 **Yeh woh moment hai jab students ka reaction best hota hai!**
> Ek full web app — sirf 30 lines of code mein

In [ ]:
# ================================================================
# STEP 6A: Gradio wrapper function
# ================================================================

def gradio_analyze(article_text):
    """
    Gradio-compatible wrapper around analyze_news pipeline.
    Returns 3 formatted strings for display in the UI.
    """
    if not article_text or len(article_text.strip()) < 50:
        return (
            "❌ Article too short!",
            "N/A",
            "Please paste a complete news article (minimum 50 characters)."
        )

    try:
        category, confidence, summary = analyze_news(article_text)

        # Category display with confidence
        category_display = f"{category}   (Confidence: {confidence})"

        # Word count stats
        original_words = len(article_text.split())
        summary_words  = len(summary.split())
        compression    = original_words / max(summary_words, 1)
        stats = (
            f"Original: {original_words} words  |  "
            f"Summary: {summary_words} words  |  "
            f"Compressed {compression:.1f}x"
        )

        return category_display, stats, summary

    except Exception as e:
        return "❌ Error", "N/A", f"Something went wrong: {str(e)}"


print("✅ Gradio wrapper function ready!")

✅ Gradio wrapper function ready!


In [ ]:
# ================================================================
# STEP 6B: Launch the Gradio Web App
# ================================================================

# Example articles jo UI mein example buttons ki tarah dikhenge
example_articles = [
    ["Apple unveiled its new M3 chip lineup bringing significant performance improvements to MacBook Pro and iMac. The chips use 3-nanometer technology making them 60 percent faster than the previous generation. Apple CEO Tim Cook called it the most powerful chip ever built for a personal computer. The chips support hardware-accelerated ray tracing and improved machine learning performance."],
    ["Real Madrid defeated Manchester City 3-2 in a thrilling UEFA Champions League quarter-final. Goals from Vinicius Junior and Rodrygo sealed the comeback win after City had led 2-0 at halftime. Goalkeeper Thibaut Courtois made several crucial saves in extra time to preserve the win for Madrid."],
    ["Amazon announced its largest ever quarterly revenue of 143 billion dollars driven by AWS cloud services and Prime membership growth. The company plans to hire 100000 new employees globally. Amazon Web Services reported 25 percent year-on-year growth as more companies migrate to cloud infrastructure."]
]

# Build the Gradio interface
demo = gr.Interface(
    fn=gradio_analyze,

    inputs=gr.Textbox(
        lines=10,
        placeholder="Yahan news article paste karo...\n\nExample: Tesla announced record profits as EV sales surged globally...",
        label="📰 News Article Input"
    ),

    outputs=[
        gr.Textbox(label="📌 Article Category",    lines=1),
        gr.Textbox(label="📊 Compression Stats",   lines=1),
        gr.Textbox(label="📝 AI Generated Summary", lines=5)
    ],

    examples=example_articles,

    title="🗞️ AI News Analyzer — Saylani AI Batch",
    description="""
### Two Transformer Models Working Together!
**DistilBERT** → Classifies the news category
**BART** → Generates a concise 2-3 line summary

Paste any English news article and see the magic! ✨
""",
    theme=gr.themes.Soft(),
    allow_flagging="never"
)

print("🚀 Launching Gradio App...")
print("   Public link bhi milega — share karke sabko access do!")

demo.launch(
    share=True,       # Public URL banata hai — students ke saath share karo
    debug=False,
    show_error=True
)

🚀 Launching Gradio App...
   Public link bhi milega — share karke sabko access do!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://872dc9e7a326b25f33.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
💡 3️⃣ Why It’s Real-World Ready

✅ No fake data — uses real pretrained models from Hugging Face

✅ Scalable — same pipeline can be connected to live news APIs

✅ Deployable — can run on Hugging Face Spaces, Streamlit Cloud, or local servers

✅ Explainable Output — classification + summary visible clearly

✅ Industry-relevant skills — students use same tools as actual NLP engineers

---
# ✍️ STEP 8: Student Practice — Apna Article Test Karo!

In [ ]:
# ================================================================
# STEP 8: Student Practice Cell
# ================================================================
# 👇 YEH TEXT APNE ARTICLE SE REPLACE KARO
# Dawn.com, BBC.com, ya kisi bhi site se ek article copy karo

YOUR_ARTICLE = """
Paste your news article here...
"""

# ── Run karo — results neeche ayenge automatically ──────────────

if "Paste your news article" in YOUR_ARTICLE:
    print("⚠️  Pehle YOUR_ARTICLE variable mein apna article paste karo!")
    print("   Upar wali string replace karo aur phir Shift+Enter se run karo.")
else:
    print("🔍 Analyzing your article...\n")
    category, confidence, summary = analyze_news(YOUR_ARTICLE)

    print("=" * 58)
    print("✅ YOUR RESULTS")
    print("=" * 58)
    print(f"📌 Category   : {category}")
    print(f"📊 Confidence : {confidence}")
    print(f"\n📝 Summary:")
    print(f"   {summary}")
    print("=" * 58)
    print(f"\n📏 Stats:")
    print(f"   Original words : {len(YOUR_ARTICLE.split())}")
    print(f"   Summary words  : {len(summary.split())}")
    print(f"   Compression    : {len(YOUR_ARTICLE.split()) / len(summary.split()):.1f}x")

---
# 📝 Class Summary — Aaj Humne Kya Seekha?

| Concept | Explanation |
|---------|-------------|
| **DistilBERT** | Encoder-only transformer — text samajhta hai, sirf label deta hai |
| **BART** | Encoder-Decoder — samajhta bhi hai, naya text generate bhi karta hai |
| **HuggingFace `pipeline()`** | 2 lines mein production-ready model |
| **Multi-Model Pipeline** | Dono models ek hi flow mein kaam kar rahe hain |
| **Gradio** | Instant web app — zero frontend code |
| **Confidence Scores** | Model kitna sure hai apne prediction se |
| **Model Comparison** | BART vs T5 — quality vs speed tradeoff |

---

# 🚀 Next Steps — Aage Kya Seekh Sakte Ho?

1. **Fine-tuning** — Is model ko Urdu news articles pe train karo
2. **More Categories** — Custom categories jodo: Crime, Health, Entertainment
3. **Web Scraping** — Directly Dawn.com se articles fetch karo using `requests` + `BeautifulSoup`
4. **FastAPI Deploy** — Is pipeline ko REST API mein convert karo
5. **Database** — Analyzed articles MongoDB ya PostgreSQL mein store karo aur search karo

